In [2]:
import os
from typing import List, Dict
import csv
from trulens.apps.custom import TruCustomApp
from trulens.core import TruSession
from trulens.core import Feedback
from trulens.providers.openai import OpenAI as tru_openai
from dotenv import load_dotenv
from trulens.apps.custom import instrument
from openai import OpenAI
import cohere



In [3]:
# with open("../../GroundTruths_Dataset - Sheet1.csv", mode='r', encoding='utf-8') as file:
#     csv_reader = csv.DictReader(file)
#     # Iterate through rows as dictionaries
#     queries = []
#     for row in csv_reader:
#         queries.append(row["query"]) 

with open("../../GroundTruths_Dataset -No Multihop No yes or no questions.csv", mode='r', encoding='utf-8') as file:
    csv_reader = csv.DictReader(file)
    # Iterate through rows as dictionaries
    queries = []
    for row in csv_reader:
        queries.append(row["query"]) 

In [4]:
len(queries)

15

In [5]:
load_dotenv()


True

In [6]:
from trulens.core import TruSession


session = TruSession()

## Uncomment the following to reset database 
# session.reset_database()

🦑 Initialized with db url sqlite:///default.sqlite .
🛑 Secret keys may be written to the database. See the `database_redact_keys` option of `TruSession` to prevent this.


In [7]:
# pip install pinecone[grpc]
from pinecone.grpc import PineconeGRPC as Pinecone


pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY_500"))
index = pc.Index("cohere-test")

In [8]:
llm = OpenAI(api_key=os.getenv("OPEN_AI_EVAL_KEY"))
co = cohere.ClientV2(os.getenv("COHERE_API_KEY"))
embed = co.embed

In [9]:
prompt ="Given the provided context, generate a response that is accurate, concise, and strictly aligned with the information retrieved. Ensure the response does not include hallucinations, speculations, or unsupported claims. The response should be neutral, fact-based, and respectful, especially when addressing sensitive or ambiguous topics. If the context provided is insufficient, clearly state that more information is needed. Prioritize safety and relevance, and avoid generating offensive or harmful content. Please the answer should be in paragraph style and sentences. do not introduce lists and bullet points"


In [10]:
class retriever:
        def __init__(self, embed, index):
             self.embed = embed
             self.index = index
        def get_data(self,query):
            embedding=self.embed( model="embed-multilingual-v3.0",texts=[query],input_type="search_query", embedding_types=["float"],).embeddings.float_[0]

            vecs = self.index.query(
            vector=embedding,
            top_k=5,
            includeMetadata=True,
            include_values=True
        )["matches"]
            ids=[] 
            for match in vecs:
                ids.append(match.id)
            data = self.index.fetch(ids)
            docs = []
            for key in data["vectors"]:
                docs.append(data["vectors"][key]["metadata"]["text"])
            return docs

In [11]:
class generator:
    def __init__(self, llm):
        self.llm = llm
    
    def generate(self, query, context):
        formatted_context = "\n".join([str(doc) for doc in context])
        response = self.llm.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": prompt},
        {
            "role": "user",
            "content": query+formatted_context
        }
    ]
)
        return response.choices[0].message

In [12]:
ret = retriever(embed, index)
gen = generator(llm)


In [ ]:
# ret.get_data("what is your mamas name")

In [13]:
class Rag_app:
    def __init__(self,llm,retriever):
        self.retriever = retriever
        self.llm=llm
    @instrument
    def retrieve(self, query: str) -> List[str]:
        """
        Method to handle document retrieval.
        IMPORTANT: The method name 'retrieve' will be used in selectors
        """
        documents = self.retriever.get_data(query)
        return documents
    
    @instrument
    def generate(self, query: str, context: List[str]) -> str:
        """
        Method to handle response generation.
        IMPORTANT: The method name 'generate' will be used in selectors
        """
        formatted_context = "\n".join([str(doc) for doc in context])
        response = self.llm.generate(query ,formatted_context)
        return response
    
    @instrument
    def query(self, question: str) -> Dict:
        """
        Main method that orchestrates the RAG pipeline.
        IMPORTANT: Return keys must match selector paths
        """
        context = self.retrieve(question)
        response = self.generate(question, context)
        
        return response.content

In [14]:
rag_app = Rag_app(gen, ret)
# provider = OpenAI(model_engine="gpt-4o", api_key=os.getenv("OPEN_AI_EVAL_KEY"))

import numpy as np
from trulens.core import Feedback
from trulens.core import Select
from trulens.providers.openai import OpenAI
provider = OpenAI()

# Define a groundedness feedback function
f_groundedness = (
    Feedback(
        provider.groundedness_measure_with_cot_reasons, name="Groundedness"
    )
    .on(Select.RecordCalls.retrieve.rets.collect())
    .on_output()
)
# Question/answer relevance between overall question and answer.
f_answer_relevance = (
    Feedback(provider.relevance_with_cot_reasons, name="Answer Relevance")
    .on_input()
    .on_output()
)

# Context relevance between question and each context chunk.
f_context_relevance = (
    Feedback(
        provider.context_relevance_with_cot_reasons, name="Context Relevance"
    )
    .on_input()
    .on(Select.RecordCalls.retrieve.rets[:])
    .aggregate(np.mean)  # choose a different aggregation method if you wish
)

✅ In Groundedness, input source will be set to __record__.app.retrieve.rets.collect() .
✅ In Groundedness, input statement will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Answer Relevance, input prompt will be set to __record__.main_input or `Select.RecordInput` .
✅ In Answer Relevance, input response will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Context Relevance, input question will be set to __record__.main_input or `Select.RecordInput` .
✅ In Context Relevance, input context will be set to __record__.app.retrieve.rets[:] .


In [15]:
from trulens.apps.custom import TruCustomApp
##NAMING CONVENTION eval-{Retriever}-{generator}-{chunksize}-@{k}

tru_rag = TruCustomApp(
    rag_app,
    app_name="NAIVE RAG",
    app_version="4o-cohere_multilingual_large_3.0-simple-500-03",
    feedbacks=[f_groundedness, f_answer_relevance, f_context_relevance],
)

In [ ]:
# rag_app.query(queries[0])

In [16]:
from trulens.core.utils.pace import Pace

# Define your desired pacing rate
pace = Pace(marks_per_second=0.5, seconds_per_period=30.0)

In [17]:
with tru_rag as recording:
    for eval in queries:
        print(eval)
        pace.mark()
        rag_app.query(
        eval
    )

Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.


How do I register for controlled or semi-controlled drugs custody?


Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.


What are the requirements for renewing the registration of a conventional pharmaceutical product?


Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.


...

How can I request a list of licensed pharmaceutical facilities in the UAE?


Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.


How do I renew my registration certificate to practice nursing or midwifery?


Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.


Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpoint. trulens will create an endpoint for cost tracking.
Could not find an instance of DummyEndpo

In [18]:
from trulens.dashboard import run_dashboard

run_dashboard(session)

Starting dashboard ...


Accordion(children=(VBox(children=(VBox(children=(Label(value='STDOUT'), Output())), VBox(children=(Label(valu…

Dashboard started at http://192.168.1.12:24727 .


<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>